# OpenAI API

In [ ]:
%pip install -q openai pandas pydantic python-dotenv

In [14]:
import asyncio
import html
import os
import re
import time
from typing import Literal, Optional
from google.colab import userdata

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import BaseModel, Field

from openai_call_metrics import (
    OpenAIMetricsSession,
    achat_completions_parse_with_metrics,
    chat_completions_create_with_metrics,
    chat_completions_parse_with_metrics,
)

# Dane

In [11]:
import zipfile
from pathlib import Path

# Nazwa pliku ZIP przesłanego do JupyterLab
zip_path = Path("real_estate_ads.csv(1).zip")

# Folder, do którego rozpakujemy archiwum
output_dir = Path("data")

output_dir.mkdir(parents=True, exist_ok=True)

# Rozpakowanie
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(output_dir)

print(f"Rozpakowano do: {output_dir.resolve()}")

# Wyświetlenie plików
for path in output_dir.rglob("*"):
    if path.is_file():
        print(f"{path}: {path.stat().st_size / 1024**2:.1f} MB")

Rozpakowano do: /content/data
data/real_estate_ads.csv: 36.2 MB
data/__MACOSX/._real_estate_ads.csv: 0.0 MB


In [12]:
plik = pd.read_csv("data/real_estate_ads.csv")
plik.head()

,id,market,created_at_first,created_at,district_lon,district_lat,title,description,params,price,no_rooms,m,price_per_m,map_lon,map_lat
0,49341,secondary,2018-01-04 13:11:05,2018-01-08 11:27:35,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>289000<br>price[curre...,289000.0,3,48.000000,6020.83,16.959516,52.391494
1,261720,secondary,2018-01-15 19:12:07,2018-01-25 11:11:31,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>379000<br>price[curre...,379000.0,3,61.000000,6213.11,16.950743,52.380535
2,50441,secondary,2018-01-04 13:53:29,2018-01-11 05:55:12,16.9408,52.38356,Rataje Nowe Miasto Niski Blok,<p>Weychan Nieruchomości Monika Hałas - Kurcza...,price<=>price<br>price<=>339000<br>price[curre...,339000.0,3,63.000000,5380.95,16.945860,52.393188
3,203518,secondary,2018-01-12 11:10:57,2018-01-30 14:38:39,16.9408,52.38356,"Mieszkanie 3 pokojowe blisko tramwaju, Poznań os.",<p>Oferuję na sprzedaż jasne i zadbane mieszka...,price<=>price<br>price<=>330000<br>price[curre...,330000.0,3,59.200001,5574.32,16.980279,52.391129
4,5,secondary,2018-01-02 11:34:51,2018-01-18 14:22:29,16.9408,52.38356,"Inwestycja -rondo Rataje student AWF,POLITECHNIKA",<p><strong>Dwa pokoje na niskim piętrze tuż pr...,price<=>price<br>price<=>259000<br>price[curre...,259000.0,2,38.000000,6815.79,16.950027,52.395861


Klucz

In [16]:
# Wczytuje OPENAI_API_KEY z pliku .env w folderze projektu.
# .env jest w .gitignore - nigdy nie commituj swojego klucza.
load_dotenv()

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "Nie znaleziono OPENAI_API_KEY. Zrób jedno z dwóch: "
        "(1) Utwórz plik .env obok tego notebooka z treścią: OPENAI_API_KEY=sk-... "
        "(2) Albo wyeksportuj klucz przed uruchomieniem Jupytera: export OPENAI_API_KEY=sk-... "
        "Notebooki w Cursor/VSCode często nie dziedziczą zmiennych wyeksportowanych w terminalu."
    )

client = OpenAI(api_key=api_key, max_retries=5)  # SDK ponawia zapytania przy limitach / chwilowych błędach

## Ustawienia, których używamy

Porównujemy dwa modele (ceny za 1M tokenów wejściowych / wyjściowych, sprawdzone 2026-10-23):

| Model | Cena | Do czego |
|---|---|---|
| `gpt-6-luna` | $0.10 / $0.50 | najnowszy tani model do dużej liczby prostych zadań |
| `gpt-5.6-luna` | $0.20 / $1.20 | poprzednia generacja, ok. 2× droższa |

- **`reasoning_effort="none"`**: nowe modele przed odpowiedzią „myślą” w ukrytych tokenach, za które płacimy. Odczytanie faktów z ogłoszenia nie wymaga myślenia, więc to wyłączamy. Oba modele mają te same ustawienia, więc porównujemy same modele.
- **`max_completion_tokens=1000`**: górny limit długości odpowiedzi, zabezpieczenie przed zbyt długą (i drogą) odpowiedzią.
- **`temperature` nie ustawiamy**: modele rozumujące zwykle akceptują tylko wartość domyślną i zwracają błąd 400 przy innej.

In [8]:
MODELS = ["gpt-6-luna", "gpt-5.6-luna"]
MAX_COMPLETION_TOKENS = 1_000

## 1. Wczytanie i czyszczenie danych

- Case dotyczy **rynku wtórnego**, więc filtrujemy po kolumnie `market`.
- **Od razu dzielimy dane na zbiór treningowy i testowy.** Na podstawie ogłoszeń wybieramy cechy i model, więc gdybyśmy oglądali przy tym ogłoszenia testowe, ocena modelu ceny na zbiorze testowym byłaby zbyt optymistyczna (wyciek danych). Podział zapisujemy do `train_test_split.csv`. Użyj tego samego podziału w swoim modelu ceny (albo wczytaj tu swój własny).
- Do porównania modeli bierzemy **losową próbkę ze zbioru treningowego**, a nie pierwsze wiersze pliku.
- Opisy są w HTML. Usunięcie tagów oszczędza tokeny (= pieniądze).

In [18]:
DATA_PATH = "real_estate_ads.csv"
SPLIT_PATH = "train_test_split.csv"
TEST_SIZE = 0.2  # 20% ogłoszeń odkładamy jako zbiór testowy
N_ADS = 10       # liczba ogłoszeń do porównania modeli (ze zbioru treningowego)

secondary = plik[(plik["market"] == "secondary") & plik["description"].notna()]

test = secondary.sample(frac=TEST_SIZE, random_state=42)
train = secondary.drop(test.index)
split = pd.DataFrame({"id": secondary["id"], "split": "train"})
split.loc[test.index, "split"] = "test"
split.to_csv(SPLIT_PATH, index=False)

sample = train.sample(n=N_ADS, random_state=42)
print(f"{len(plik)} ogłoszeń łącznie, {len(secondary)} z rynku wtórnego: "
      f"{len(train)} train / {len(test)} test, {len(sample)} w próbce do porównania")


def clean_description(raw: str, max_chars: int = 4000) -> str:
    """Zamienia opis w HTML na zwykły tekst (i przycina bardzo długie opisy)."""
    text = html.unescape(str(raw))
    text = re.sub(r"<br\s*/?>|</p>|</li>", "\n", text)  # zachowujemy podział na linie
    text = re.sub(r"<[^>]+>", " ", text)                 # usuwamy pozostałe tagi
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()[:max_chars]


print(clean_description(sample["description"].iloc[0]))

13745 ogłoszeń łącznie, 5954 z rynku wtórnego: 4763 train / 1191 test, 10 w próbce do porównania
BEZPIECZNA TRANSAKCJA JEDYNIE Z DOŚWIADCZONYM I UBEZPIECZONYM POŚREDNIKIEM!!!
Mam przyjemność zaprezentować Państwu mieszkanie 2 pokojowe położone na Ratajach ( os. Czecha ).
Nieruchomość o powierzchni ok. 43 m2 znajduję się na DZIESIĄTYM PIĘTRZE W WYSOKIM BLOKU Z WINDĄ.
Mieszkanie składa się z:
- salonu z wyjściem na balkon typu loggia od strony zachodniej,
- sypialni od strony wschodniej,
- osobnej kuchni z oknem od strony wschodniej,
- korytarza,
- łazienki oraz osobnego wc.
UKŁAD POMIESZCZEŃ NIEPRZECHODNI!!! MIESZKANIE DWUSTRONNE!!!
Nowe okna PCV.
W pokojach na podłogach panele.
W łazience, kuchni, WC i na korytarzu płytki.
Forma własności: własnściowo - spółdzielcze.
Opłaty czynszowe ok 350 zł.
Wolne do uzgodnienia.
Do mieszkania przynależy piwnica.
 Oferta skierowana dla klientów z gotówką.
Cena ***-***-*** zł do negocjacji!!!
Zapraszam na prezentację i składanie ofert!!!
Agent prowad

## 2. Pierwsze wywołanie: co jest w ogłoszeniach?

Zwykłe zapytanie z odpowiedzią tekstową. Dajemy modelowi 20 ogłoszeń **ze zbioru treningowego** i listę kolumn, które już mamy, i prosimy o propozycję **nowych** cech. Odpowiedź czytamy sami i na jej podstawie wybieramy cechy do schematu w kroku 3.

Wiadomość ma dwie role:
- `system`: instrukcja, jak model ma się zachować,
- `user`: dane, na których ma pracować.

In [19]:
# Kolumny, które już mamy: klucze z `params` plus główne kolumny zbioru danych.
params_keys = sorted({
    kv.split("<=>")[0]
    for p in train["params"].dropna()
    for kv in p.split("<br>") if "<=>" in kv
})
known_columns = ["m", "no_rooms", "price", "map_lat", "map_lon"] + params_keys

ads_text = "\n\n".join(
    f"--- OGŁOSZENIE {i + 1} ---\n{clean_description(d, max_chars=1500)}"
    for i, d in enumerate(train.drop(sample.index).sample(n=20, random_state=0)["description"])  # tylko train!
)

explore_metrics = OpenAIMetricsSession()
response = chat_completions_create_with_metrics(
    client,
    session=explore_metrics,
    model="gpt-6-luna",
    messages=[
        {
            "role": "system",
            "content": (
                "You help build a model that predicts the price per m2 of apartments in Poznań. "
                "Read the ads below and propose 10-15 features that can be read from the description text, "
                "are likely to affect the price per m2 and appear in many ads. "
                f"Do NOT propose features already available as columns: {', '.join(known_columns)}. "
                "For each feature give an English snake_case name and one sentence in Polish explaining it. "
                "Answer in Polish."
            ),
        },
        {"role": "user", "content": ads_text},
    ],
    reasoning_effort="low",  # tu model ma porównać 20 ogłoszeń, więc pozwalamy mu trochę pomyśleć
    max_completion_tokens=4_000,
)
print(response.choices[0].message.content)
print()
print(explore_metrics.format_summary())

1. `renovation_condition` — Opisuje stan lokalu i zakres lub świeżość remontu, np. mieszkanie do remontu, zadbane albo niedawno wyremontowane.
2. `finish_standard` — Określa deklarowany standard wykończenia wnętrza, od deweloperskiego po podwyższony lub luksusowy.
3. `layout_functionality` — Informuje, czy pokoje są niezależne, przechodnie lub połączone z kuchnią, co wpływa na funkcjonalność mieszkania.
4. `room_layout_flexibility` — Wskazuje, czy układ można łatwo zmienić, np. wydzielić dodatkowy pokój lub zrobić aneks kuchenny.
5. `window_exposure` — Rejestruje opisaną orientację okien względem stron świata, która wpływa m.in. na nasłonecznienie.
6. `view_quality` — Opisuje widok z okien, np. na zieleń, park, panoramę miasta, sąsiedni budynek lub ruchliwą ulicę.
7. `balcony_area_sqm` — Zapisuje podaną w ogłoszeniu powierzchnię balkonu, loggii lub tarasu, jeśli została wymieniona.
8. `storage_space_availability` — Informuje, czy do mieszkania przynależy piwnica, komórka lokatorska lub

## 3. Schemat: jakie cechy chcemy dostać

Odpowiedź tekstowa za każdym razem wygląda trochę inaczej, a do modelu ceny potrzebujemy **tych samych kolumn dla każdego ogłoszenia**. Dlatego opisujemy cechy jako klasę Pydantic i przekazujemy ją do API (Structured Outputs). Model musi wtedy zwrócić dokładnie taki format.

- `Literal[...]`: model musi wybrać jedną z podanych wartości (cecha kategoryczna).
- `Optional[...]`: model może zwrócić `None`, gdy ogłoszenie o czymś nie wspomina. Chcemy *braku danych*, a nie zgadywania.
- `description=`: trafia do modelu, więc działa jak instrukcja (dlatego po angielsku).

**Wybrane cechy**


*   renovation_condition
*   finish_standard
*   layout_functionality
*   window_exposure
*   view_quality
*   balcony_area_sqm
*   num_bathrooms
*   has_elevators
*   has_storage
*   near_public_transport
*   sold_by



In [20]:
class ApartmentFeatures(BaseModel):
    renovation_condition: Optional[Literal["to_renovate", "to_refresh", "good", "renovated", "high_standard"]] = Field(
        description="Condition of the apartment as described in the ad."
    )
    finish_standard: Optional[Literal["shell_and_core", "standard", "high_standard", "luxury"]]= Field(
        description="Declared interior finish standard, from developer state to high-standard and luxury."
    )
    layout_functionality: Optional[Literal["separate", "walk_through", "open_kitchen"]] = Field(
        description="Whether the rooms are separate, walk-through or connected to kitchen."
    )
    window_exposure: Optional[str] = Field(
        description="Described orientation of windows relative to cardinal directions, affecting sunlight."
    )
    view_quality: Optional[Literal["greenery", "park", "city_panorama", "neighbouring_building", "busy_street"]] = Field(
        description="View from windows, e.g., greenery, park, city panorama, neighbouring building, or busy street."
    )
    balcony_area_sqm: Optional[float] = Field(
        description="Area of the balcony, loggia, or terrace in square meters, if mentioned in the ad."
    )
    num_bathrooms: Optional[int] = Field(description="Number of bathrooms, if stated.")
    parking: Optional[Literal["none", "street", "parking_spot", "garage"]] = Field(
        description="Best parking option included or mentioned for the apartment."
    )
    has_elevator: Optional[bool] = Field(description="Building has an elevator (winda).")
    has_storage: Optional[bool] = Field(description="Basement or storage room (piwnica, komórka lokatorska).")
    near_public_transport: Optional[bool] = Field(description="Tram/bus stops mentioned as nearby.")
    sold_by: Optional[Literal["agency", "private"]] = Field(
        description="Agency if a real estate agent/office/licence is mentioned, private if the owner sells directly."
    )

## 4. Ekstrakcja tymi samymi ogłoszeniami w obu modelach

Zamiast `create` używamy `client.chat.completions.parse(...)` z `response_format=ApartmentFeatures`. Gotowy obiekt jest w `message.parsed`, nie trzeba samemu parsować tekstu. Każdy model ma osobny licznik kosztów, żeby łatwo je porównać.

In [21]:
SYSTEM_PROMPT = """You are a data extraction assistant for real estate listings.
You receive a Polish apartment-for-sale ad from Poznań (secondary market).
Extract the requested fields using ONLY information stated in the ad.
If something is not mentioned, return null - never guess."""


def build_messages(description: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": clean_description(description)},
    ]


def extract_features(description: str, model: str, metrics: OpenAIMetricsSession) -> ApartmentFeatures:
    response = chat_completions_parse_with_metrics(
        client,
        session=metrics,
        model=model,
        messages=build_messages(description),
        response_format=ApartmentFeatures,
        reasoning_effort="none",
        max_completion_tokens=MAX_COMPLETION_TOKENS,
    )
    return response.choices[0].message.parsed

In [22]:
# Zabezpieczenie przed przypadkowym uruchomieniem porównania na dużej liczbie ogłoszeń.
assert N_ADS <= 100, "Porównanie modeli rób na maks. 100 ogłoszeniach; cały zbiór przetwarzasz w kroku 6."

results = {}  # model -> DataFrame z cechami
metrics = {}  # model -> licznik kosztów i czasu

for model in MODELS:
    metrics[model] = OpenAIMetricsSession()
    rows = []
    for ad_id, description in zip(sample["id"], sample["description"]):
        try:
            features = extract_features(description, model, metrics[model])
        except Exception as e:  # jedno problematyczne ogłoszenie nie powinno zatrzymać całej pętli
            print(f"{model} id={ad_id} błąd: {e}")
            continue
        rows.append({"id": ad_id, **features.model_dump()})
    results[model] = pd.DataFrame(rows).set_index("id")
    print(f"{model}: gotowe, {len(rows)} ogłoszeń")

results[MODELS[0]]

gpt-6-luna: gotowe, 10 ogłoszeń
gpt-5.6-luna: gotowe, 10 ogłoszeń


,renovation_condition,finish_standard,layout_functionality,window_exposure,view_quality,balcony_area_sqm,num_bathrooms,parking,has_elevator,has_storage,near_public_transport,sold_by
id,,,,,,,,,,,,
2901901,None,None,separate,"west (living room and loggia), east (bedroom a...",None,NaN,1.0,None,True,True,None,agency
3959301,good,None,None,wschodnia i południowa,greenery,NaN,NaN,street,None,True,True,private
4406553,to_renovate,None,separate,None,None,NaN,1.0,None,None,True,True,agency
8445790,None,None,None,None,None,7.0,NaN,garage,None,True,True,agency
4976361,to_refresh,None,separate,None,None,NaN,1.0,None,None,True,True,agency
9152924,to_refresh,None,None,None,None,NaN,1.0,None,True,True,True,agency
1574693,renovated,high_standard,open_kitchen,None,None,NaN,1.0,None,None,None,True,agency
1908005,to_refresh,None,open_kitchen,None,city_panorama,NaN,2.0,garage,True,None,True,agency
4881037,to_refresh,None,open_kitchen,east,greenery,NaN,1.0,None,None,True,True,agency


## 5. Porównanie modeli

### Zgodność odpowiedzi
Jak często oba modele dały tę samą odpowiedź (brak danych w obu też liczymy jako zgodność)?

In [23]:
model_a, model_b = MODELS
a, b = results[model_a], results[model_b]
common = a.index.intersection(b.index)
a, b = a.loc[common], b.loc[common]

agreement = ((a == b) | (a.isna() & b.isna())).mean().round(2)
agreement.rename("zgodność")

,zgodność
renovation_condition,0.8
finish_standard,0.7
layout_functionality,0.8
window_exposure,0.8
view_quality,1.0
balcony_area_sqm,1.0
num_bathrooms,0.9
parking,1.0
has_elevator,0.9
has_storage,1.0


Zgodności dla wszystkich cech są wysokie (najniższe 0.7 dla finish_standard). Można się zastanowić nad jasnością opisu cech finish_standard, layout_functionality, window_exposure i renovation_quality. Dla pozostałych cech modele bardzo zgodne.

In [24]:
# Tylko ogłoszenia i cechy, w których modele się różnią.
a.compare(b, result_names=(model_a, model_b))

renovation_condition              finish_standard               \
                  gpt-6-luna gpt-5.6-luna      gpt-6-luna gpt-5.6-luna   
id                                                                       
2901901                 None         good            None     standard   
3959301                  NaN          NaN             NaN          NaN   
4976361                  NaN          NaN            None     standard   
9152924           to_refresh    renovated             NaN          NaN   
4881037                  NaN          NaN             NaN          NaN   
7608665                  NaN          NaN            None     standard   

        layout_functionality               \
                  gpt-6-luna gpt-5.6-luna   
id                                          
2901901                  NaN          NaN   
3959301                 None     separate   
4976361                  NaN          NaN   
9152924                 None     separate   
4881037                  NaN          NaN   
7608665                  NaN          NaN   

                                           window_exposure  \
                                                gpt-6-luna   
id                                                           
2901901  west (living room and loggia), east (bedroom a...   
3959301                                                NaN   
4976361                                                NaN   
9152924                                                NaN   
4881037                                               east   
7608665                                                NaN   

                               num_bathrooms              has_elevator  \
                  gpt-5.6-luna    gpt-6-luna gpt-5.6-luna   gpt-6-luna   
id                                                                       
2901901  zachodnia i wschodnia           NaN          NaN          NaN   
3959301                    NaN           NaN          1.0          NaN   
4976361                    NaN           NaN          NaN         None   
9152924                    NaN           NaN          NaN          NaN   
4881037              wschodnia           NaN          NaN          NaN   
7608665                    NaN           NaN          NaN          NaN   

                      
        gpt-5.6-luna  
id                    
2901901          NaN  
3959301          NaN  
4976361         True  
9152924          NaN  
4881037          NaN  
7608665          NaN

### Koszt i czas
Przy `reasoning_effort="none"` kolumna `reasoning` powinna mieć 0. Ostatnia kolumna szacuje koszt dla **wszystkich** ogłoszeń z rynku wtórnego.

In [25]:
summary = []
for model, m in metrics.items():
    n = len(m.calls)
    summary.append({
        "model": model,
        "ogłoszeń": n,
        "śr. tokeny in": round(m.total_input_tokens / n),
        "śr. tokeny out": round(m.total_output_tokens / n),
        "śr. reasoning": round(sum(c.reasoning_tokens for c in m.calls) / n),
        "śr. czas [s]": round(m.total_elapsed_s / n, 2),
        "koszt [$]": round(m.total_cost_usd, 4),
        f"cały rynek wtórny ({len(secondary)}) [$]": round(m.total_cost_usd / n * len(secondary), 2),
    })
pd.DataFrame(summary).set_index("model")

,ogłoszeń,śr. tokeny in,śr. tokeny out,śr. reasoning,śr. czas [s],koszt [$],cały rynek wtórny (5954) [$]
model,,,,,,,
gpt-6-luna,10,1222,83,0,2.65,0.0016,0.97
gpt-5.6-luna,10,1222,83,0,1.59,0.0034,2.05


## 6. Cały zbiór wybranym modelem (asyncio)

Dla wybranego modelu gpt-6-luna przetwarzamy wszystkie ogłoszenia z rynku wtórnego, treningowe i testowe. To nie jest wyciek danych: ekstrakcja odczytuje fakty z każdego ogłoszenia osobno i nie korzysta z ceny.

Wysyłanie zapytań po kolei trwałoby 2–3 godziny, bo prawie cały czas czekamy na odpowiedź serwera. Z `asyncio` wysyłamy wiele zapytań naraz:
- `AsyncOpenAI` to asynchroniczna wersja klienta, a jego metody wywołujemy z `await`,
- `asyncio.Semaphore(CONCURRENCY)` pilnuje, żeby naraz leciało maksymalnie `CONCURRENCY` zapytań (limit dla `gpt-6-luna` to 5000 zapytań/min),
- `asyncio.gather(...)` uruchamia wszystkie zadania z paczki i czeka, aż się skończą,
- w Jupyterze piszemy po prostu `await ...` w komórce (`asyncio.run()` w notebooku nie działa).

Wyniki zapisujemy do `llm_features.csv` co `CHUNK_SIZE` ogłoszeń. Jeśli coś przerwie działanie, ponowne uruchomienie pominie gotowe ogłoszenia, więc nie płacisz dwa razy.

In [29]:
SELECTED_MODEL = "gpt-6-luna"  # model wybrany na podstawie porównania z kroku 5
CONCURRENCY = 16               # ile zapytań naraz
CHUNK_SIZE = 500               # co tyle ogłoszeń zapisujemy wyniki do pliku
OUTPUT_PATH = "llm_features.csv"
RUN_FULL = True               # zmień na True, żeby uruchomić (ok. $1, kilka-kilkanaście minut)

done_ids = set(pd.read_csv(OUTPUT_PATH)["id"]) if os.path.exists(OUTPUT_PATH) else set()
todo = secondary[~secondary["id"].isin(done_ids)]
print(f"Gotowe: {len(done_ids)}, do zrobienia: {len(todo)} ogłoszeń")
if metrics.get(SELECTED_MODEL) and metrics[SELECTED_MODEL].calls:
    per_ad = metrics[SELECTED_MODEL].total_cost_usd / len(metrics[SELECTED_MODEL].calls)
    print(f"Szacowany koszt (na podstawie kroku 5): ~${per_ad * len(todo):.2f}")

Gotowe: 0, do zrobienia: 5954 ogłoszeń
Szacowany koszt (na podstawie kroku 5): ~$0.97


In [30]:
async_client = AsyncOpenAI(api_key=api_key, max_retries=5)
semaphore = asyncio.Semaphore(CONCURRENCY)
full_metrics = OpenAIMetricsSession()


async def extract_row(ad_id, description) -> Optional[dict]:
    async with semaphore:  # czekamy, aż zwolni się miejsce (maks. CONCURRENCY zapytań naraz)
        try:
            response = await achat_completions_parse_with_metrics(
                async_client,
                session=full_metrics,
                model=SELECTED_MODEL,
                messages=build_messages(description),
                response_format=ApartmentFeatures,
                reasoning_effort="none",
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )
            return {"id": ad_id, **response.choices[0].message.parsed.model_dump()}
        except Exception as e:  # jedno problematyczne ogłoszenie nie powinno zatrzymać całości
            print(f"id={ad_id} błąd: {e}")
            return None


if RUN_FULL:
    t0 = time.perf_counter()
    for start in range(0, len(todo), CHUNK_SIZE):
        chunk = todo.iloc[start:start + CHUNK_SIZE]
        rows = await asyncio.gather(*(extract_row(i, d) for i, d in zip(chunk["id"], chunk["description"])))
        rows = [r for r in rows if r is not None]
        if rows:
            pd.DataFrame(rows).to_csv(OUTPUT_PATH, mode="a", header=not os.path.exists(OUTPUT_PATH), index=False)
        print(f"{start + len(chunk)}/{len(todo)} ogłoszeń, {time.perf_counter() - t0:.0f} s")
    print(f"\nCzas całości: {time.perf_counter() - t0:.0f} s "
          f"(suma czasów pojedynczych zapytań: {full_metrics.total_elapsed_s:.0f} s)")
    print(f"Koszt: ${full_metrics.total_cost_usd or 0:.2f}")
else:
    print("RUN_FULL = False - nic nie zostało wysłane.")

500/5954 ogłoszeń, 88 s
1000/5954 ogłoszeń, 164 s
1500/5954 ogłoszeń, 238 s
2000/5954 ogłoszeń, 303 s
2500/5954 ogłoszeń, 373 s
3000/5954 ogłoszeń, 442 s
3500/5954 ogłoszeń, 516 s
4000/5954 ogłoszeń, 581 s
4500/5954 ogłoszeń, 644 s
5000/5954 ogłoszeń, 715 s
5500/5954 ogłoszeń, 787 s
5954/5954 ogłoszeń, 843 s

Czas całości: 843 s (suma czasów pojedynczych zapytań: 13024 s)
Koszt: $0.95


### Połączenie z podziałem train/test
Każdy wiersz ma teraz informację, czy należy do zbioru treningowego, czy testowego. Trenuj model ceny na `train`, a `test` zostaw na końcową ocenę.

In [31]:
if os.path.exists(OUTPUT_PATH):
    llm_features = pd.read_csv(OUTPUT_PATH).merge(split, on="id")
    print(llm_features["split"].value_counts())
    display(llm_features.head())

split
train    4763
test     1191
Name: count, dtype: int64


,id,renovation_condition,finish_standard,layout_functionality,window_exposure,view_quality,balcony_area_sqm,num_bathrooms,parking,has_elevator,has_storage,near_public_transport,sold_by,split
0,49341,NaN,NaN,separate,NaN,NaN,NaN,1.0,NaN,NaN,NaN,True,agency,train
1,261720,NaN,high_standard,open_kitchen,NaN,NaN,NaN,1.0,NaN,NaN,True,NaN,agency,train
2,50441,to_refresh,NaN,separate,east and west,greenery,NaN,NaN,NaN,NaN,True,True,agency,train
3,203518,to_refresh,NaN,separate,NaN,greenery,NaN,NaN,NaN,NaN,True,True,agency,train
4,5,NaN,NaN,open_kitchen,east,NaN,NaN,1.0,street,NaN,True,True,agency,train


In [35]:
df_main = plik
df_llm = pd.read_csv("llm_features.csv")

# Połączenie tabel po kolumnie 'id'
merged_df = pd.merge(df_main, df_llm, on="id", how="inner")

print(f"Liczba wierszy po połączeniu: {merged_df.shape[0]}")
print(f"Liczba kolumn po połączeniu: {merged_df.shape[1]}")
display(merged_df.head(3))

Liczba wierszy po połączeniu: 5954
Liczba kolumn po połączeniu: 27


,id,market,created_at_first,created_at,district_lon,district_lat,title,description,params,price,...,layout_functionality,window_exposure,view_quality,balcony_area_sqm,num_bathrooms,parking,has_elevator,has_storage,near_public_transport,sold_by
0,49341,secondary,2018-01-04 13:11:05,2018-01-08 11:27:35,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>289000<br>price[curre...,289000.0,...,separate,NaN,NaN,NaN,1.0,NaN,NaN,NaN,True,agency
1,261720,secondary,2018-01-15 19:12:07,2018-01-25 11:11:31,16.9408,52.38356,NaN,<p>Zapraszam do zapoznania się z ofertą sprzed...,price<=>price<br>price<=>379000<br>price[curre...,379000.0,...,open_kitchen,NaN,NaN,NaN,1.0,NaN,NaN,True,NaN,agency
2,50441,secondary,2018-01-04 13:53:29,2018-01-11 05:55:12,16.9408,52.38356,Rataje Nowe Miasto Niski Blok,<p>Weychan Nieruchomości Monika Hałas - Kurcza...,price<=>price<br>price<=>339000<br>price[curre...,339000.0,...,separate,east and west,greenery,NaN,NaN,NaN,NaN,True,True,agency


In [36]:
merged_df.to_csv("real_estate_with_llm_features.csv", index=False)